# Final benchmark, part 08b — **100 problems**: `dense500` and `dense1500` (dense SFT, all 3 stages each)
### Run this alongside notebook 08a; together they cover all 10 configurations

This is one half of the final n=100 benchmark, split so the two halves can run **in parallel on
two separate GPUs**. This half covers 6 model configurations at t=1 and t=2.

| Arm | Checkpoints | Source run |
|---|---|---|
| `dense500` | s1_t4, s2_t2, s3_t1 | `sdar_superviseall` (notebook 02) |
| `dense1500` | s1_t4, s2_t2, s3_t1 | `sdar_dense_matched` (notebook 07) |

**6 configs x 2 schedules x 100 problems = 1200 rollouts, ~23.3 h on one GPU.**

> ### Splitting only helps if you run the two halves on DIFFERENT GPUs
> On a single GPU the total work is unchanged (~39 h across both halves). The benefit is
> wall-clock when 08a and 08b run simultaneously in two Colab sessions.
>
> Both halves are checkpointed per rollout and resume automatically, so either can be stopped
> and restarted freely.

### Two things that keep the halves combinable

**1. Identical problems.** Both notebooks read the same cached `bench100.json` and print a
`BENCH HASH`. **The hash must match in both.** Section 5 asserts this against a stored
`bench100.hash`, so a mismatch stops the run rather than silently producing incomparable
numbers. If you have not run either half yet, start whichever you like: the first to reach §5
builds the cache and the second verifies against it.

**2. Separate output files.** This half writes `results/rows_dense.jsonl`; the other writes its own.
Two Colab sessions appending to one file on Drive will interleave writes and corrupt it, so
they are deliberately kept apart. The analysis sections glob **all** `rows_*.jsonl` and merge
them, so whichever half you run §9 in will show everything recorded so far, and will name any
configurations still missing.

**Per-problem records are saved**, so §9 computes real bootstrap confidence intervals and §10
runs the per-stage significance tests that were underpowered at n=40.

## 1 · GPU & Drive

In [ ]:
!nvidia-smi
import torch, platform
print("\nTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| Python:", platform.python_version())
assert torch.cuda.is_available(), "No GPU — Runtime ▸ Change runtime type."
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2 · Dependencies — torchao removed **before** transformers is imported
**If the transformers version changes, restart the runtime and re-run from the top.**

In [ ]:
import os, re, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
for _n in [n for n in list(sys.modules) if n == "torchao" or n.startswith("torchao.")]:
    del sys.modules[_n]

if not os.path.isdir('/content/dLLM-RL'):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/Gen-Verse/dLLM-RL","/content/dLLM-RL"], check=True)

_FALLBACK = "transformers==4.51.3"; _spec = _FALLBACK
_req = "/content/dLLM-RL/requirements.txt"
if os.path.isfile(_req):
    m = re.search(r"^\s*transformers(\[[^\]]*\])?\s*([=<>!~].*?)\s*(?:#.*)?$", open(_req).read(), re.M)
    if m and m.group(2): _spec = "transformers" + (m.group(1) or "") + m.group(2).strip()
print("Pinning:", _spec)

!pip -q install "{_spec}" "accelerate>=0.33" "peft>=0.12" "datasets>=2.20" sentencepiece packaging

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
print("torchao present?", importlib.util.find_spec("torchao") is not None, "(must be False)")
import transformers; print("transformers:", transformers.__version__)
print("⚠️ If that version just CHANGED: Runtime ▸ Restart session, then run from the top.")


## 4 · `flash_attn` → pure-PyTorch replacements

In [ ]:
import os, sys, types, importlib, importlib.abc, importlib.machinery, importlib.util
import torch, torch.nn.functional as F

USE_FLASH_ATTN = False
for _n in [n for n in list(sys.modules) if n=="flash_attn" or n.startswith("flash_attn.")]:
    del sys.modules[_n]
for _n in [n for n in list(sys.modules) if "modeling_sdar" in n]:
    del sys.modules[_n]

import transformers.dynamic_module_utils as _dmu
_real = getattr(_dmu, "_orig_get_imports", _dmu.get_imports)
_dmu._orig_get_imports = _real
def _pgi(fn):
    imp = list(_real(fn))
    if "flash_attn" in imp and os.path.basename(str(fn)).startswith("modeling_"):
        imp = [i for i in imp if i != "flash_attn"]
    return imp
_dmu.get_imports = _pgi

import transformers.utils as _tu
for _old, _c in {"LossKwargs": ["TransformersKwargs"]}.items():
    if not hasattr(_tu, _old):
        v = None
        for cand in _c:
            for mp in ("transformers.utils","transformers.processing_utils",
                       "transformers.modeling_utils","transformers"):
                try:
                    m = importlib.import_module(mp)
                    if hasattr(m, cand): v = getattr(m, cand); break
                except Exception: pass
            if v is not None: break
        setattr(_tu, _old, v if v is not None else type(_old, (dict,), {}))

def _rms_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                 eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                 zero_centered_weight=False, return_dropout_mask=False, out_dtype=None,
                 out=None, residual_out=None):
    xdt = x.dtype
    if x1 is not None: x = x + x1
    base = ((x.float()+residual.float()) if residual_in_fp32 else (x+residual)) if residual is not None \
           else (x.float() if residual_in_fp32 else x)
    nr = base; xf = base.float()
    y = (xf*torch.rsqrt(xf.pow(2).mean(-1,keepdim=True)+eps)).to(xdt) * \
        ((1.0+weight) if zero_centered_weight else weight)
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _layer_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                   eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                   zero_centered_weight=False, is_rms_norm=False, return_dropout_mask=False,
                   out_dtype=None, out=None, residual_out=None):
    if is_rms_norm:
        return _rms_norm_fn(x, weight, bias, residual, x1, weight1, bias1, eps, dropout_p,
                            rowscale, prenorm, residual_in_fp32, zero_centered_weight,
                            return_dropout_mask, out_dtype, out, residual_out)
    xdt = x.dtype
    if x1 is not None: x = x + x1
    base = ((x.float()+residual.float()) if residual_in_fp32 else (x+residual)) if residual is not None \
           else (x.float() if residual_in_fp32 else x)
    nr = base; xf = base.float(); mu = xf.mean(-1,keepdim=True)
    y = ((xf-mu)*torch.rsqrt((xf-mu).pow(2).mean(-1,keepdim=True)+eps)).to(xdt) * \
        ((1.0+weight) if zero_centered_weight else weight)
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _expand_kv(k,v,nq):
    nk = k.shape[-2]
    if nk != nq:
        r = nq//nk; k = k.repeat_interleave(r,dim=-2); v = v.repeat_interleave(r,dim=-2)
    return k,v
def _flash_attn_func(q,k,v,dropout_p=0.0,softmax_scale=None,causal=False,window_size=(-1,-1),
                     softcap=0.0,alibi_slopes=None,deterministic=False,return_attn_probs=False,**kw):
    k,v=_expand_kv(k,v,q.shape[-2])
    o=F.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),
                                     is_causal=causal,scale=softmax_scale,dropout_p=0.0)
    return o.transpose(1,2)
def _flash_attn_qkvpacked_func(qkv,**kw):
    q,k,v=qkv.unbind(dim=2); return _flash_attn_func(q,k,v,**kw)
def _flash_attn_varlen_func(q,k,v,cu_seqlens_q,cu_seqlens_k,max_seqlen_q=None,max_seqlen_k=None,
                            dropout_p=0.0,softmax_scale=None,causal=False,**kw):
    cq,ck_=cu_seqlens_q.tolist(),cu_seqlens_k.tolist(); outs=[]
    for i in range(len(cq)-1):
        qi,ki,vi=q[cq[i]:cq[i+1]],k[ck_[i]:ck_[i+1]],v[ck_[i]:ck_[i+1]]
        ki,vi=_expand_kv(ki,vi,qi.shape[-2])
        oi=F.scaled_dot_product_attention(qi.transpose(0,1).unsqueeze(0),ki.transpose(0,1).unsqueeze(0),
                                          vi.transpose(0,1).unsqueeze(0),is_causal=causal,
                                          scale=softmax_scale,dropout_p=0.0)
        outs.append(oi.squeeze(0).transpose(0,1))
    return torch.cat(outs,0)
def _pad_input(hs,idx,b,s):
    out=hs.new_zeros(b*s,hs.shape[-1]); out[idx]=hs; return out.view(b,s,-1)
def _unpad_input(hs,am,*a,**k):
    sl=am.sum(-1).to(torch.int32); idx=torch.nonzero(am.flatten(),as_tuple=False).flatten()
    h=hs.reshape(-1,hs.shape[-1])[idx]; cu=torch.zeros(sl.numel()+1,dtype=torch.int32,device=hs.device)
    cu[1:]=torch.cumsum(sl,0); return h,idx,cu,int(sl.max().item())
def _index_first_axis(x,idx): return x.reshape(-1,*x.shape[1:])[idx]

class _RMSNormModule(torch.nn.Module):
    def __init__(self,hidden_size,eps=1e-6,**kw):
        super().__init__(); self.weight=torch.nn.Parameter(torch.ones(hidden_size)); self.eps=eps
    def forward(self,x,residual=None,prenorm=False,**kw):
        return _rms_norm_fn(x,self.weight,None,residual=residual,eps=self.eps,prenorm=prenorm)

_REG={"rms_norm_fn":_rms_norm_fn,"layer_norm_fn":_layer_norm_fn,"RMSNorm":_RMSNormModule,
      "LayerNorm":torch.nn.LayerNorm,"flash_attn_func":_flash_attn_func,
      "flash_attn_qkvpacked_func":_flash_attn_qkvpacked_func,
      "flash_attn_varlen_func":_flash_attn_varlen_func,"pad_input":_pad_input,
      "unpad_input":_unpad_input,"index_first_axis":_index_first_axis}
def _uns(n):
    def f(*a,**k): raise RuntimeError(f"flash_attn.{n} has no shim but was CALLED — report it.")
    return f
class _FM(types.ModuleType):
    def __getattr__(self,n):
        if n in _REG: return _REG[n]
        if n.startswith("__"): raise AttributeError(n)
        return _uns(n)
class _FF(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self,fn,path=None,target=None):
        if fn=="flash_attn" or fn.startswith("flash_attn."):
            return importlib.machinery.ModuleSpec(fn,self,is_package=True)
    def create_module(self,spec):
        m=_FM(spec.name); m.__spec__=spec; m.__path__=[]; m.__version__="0.0-shim"; return m
    def exec_module(self,m): pass

try:
    import flash_attn; USE_FLASH_ATTN=True; print("Real flash_attn present.")
except ImportError:
    if not any(isinstance(f,_FF) for f in sys.meta_path): sys.meta_path.insert(0,_FF())
    import flash_attn; print("✅ pure-PyTorch flash_attn replacements active.")
_t=torch.randn(2,4,8); _w=torch.randn(8)
assert torch.allclose(_rms_norm_fn(_t,_w,eps=1e-6),
                      _t*torch.rsqrt(_t.pow(2).mean(-1,keepdim=True)+1e-6)*_w, atol=1e-5)
print("✅ RMSNorm replacement matches reference math.")


## 5 · Metrics

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

def repeat4(text):
    t = text.split()
    if len(t) < 4: return 0.0
    g = [tuple(t[i:i+4]) for i in range(len(t)-3)]
    return 1.0 - len(set(g))/len(g)

def extract_boxed(text):
    if not text: return None
    i = text.rfind("\\boxed")
    if i == -1: return None
    j = text.find("{", i)
    if j == -1: return None
    d = 0
    for k in range(j, len(text)):
        if text[k] == "{": d += 1
        elif text[k] == "}":
            d -= 1
            if d == 0: return text[j+1:k]
    return None

def has_complete_box(t): return extract_boxed(t) is not None

def _norm(s):
    if s is None: return None
    s = str(s).strip().replace(" ","")
    for a,b in (("\\left",""),("\\right",""),("\\dfrac","\\frac"),("\\tfrac","\\frac"),("$","")):
        s = s.replace(a,b)
    if s.startswith("\\text{") and s.endswith("}"): s = s[6:-1]
    return s

def answers_match(pred, gold):
    a,b = _norm(pred), _norm(gold)
    if a is None or b is None: return False
    if a == b: return True
    try: return abs(float(a)-float(b)) < 1e-6
    except Exception: return False

def structure_metrics(gen, ref, n_chunks=6):
    w = gen.split()
    if len(w) < n_chunks*4 or not ref.strip(): return float("nan"),float("nan"),float("nan")
    chunks = [" ".join(c) for c in np.array_split(np.array(w), n_chunks)]
    rw = ref.split(); tail = " ".join(rw[-max(20,len(rw)//4):])
    corpus = chunks + [tail, ref, gen]
    try: V = TfidfVectorizer(ngram_range=(1,2), min_df=1).fit_transform(corpus)
    except ValueError: return float("nan"),float("nan"),float("nan")
    C,tl,fr,fg = V[:n_chunks],V[n_chunks],V[n_chunks+1],V[n_chunks+2]
    sims = cosine_similarity(C,tl).ravel()
    prog = spearmanr(np.arange(n_chunks),sims).correlation if float(np.std(sims))>1e-9 else 0.0
    if prog is None or np.isnan(prog): prog = 0.0
    P = cosine_similarity(C); iu = np.triu_indices(n_chunks,k=1)
    return float(prog), float(P[iu].mean()), float(cosine_similarity(fg,fr)[0,0])

assert extract_boxed(r"a \boxed{1}, b \boxed{\frac{1}{2}}") == r"\frac{1}{2}"
assert answers_match(r"\dfrac{1}{2}", r"\frac{1}{2}") and not answers_match("3","4")
assert repeat4("a b c d "*6) > 0.5
print("✅ metric self-tests pass")


## 3 · Config — point these at your three training runs

`drive_roots` must match where each run actually wrote its adapters. The §5 preflight check
verifies every adapter exists **before** any GPU time is spent, so a typo costs seconds instead
of hours.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
import os

@dataclass
class Cfg:
    out_root: str = "/content/drive/MyDrive/sdar_bench100"

    # ---- where each training run wrote its adapters (ckpt/<stage>/) ----
    drive_roots: Dict[str, str] = field(default_factory=lambda: {
        "masked":    "/content/drive/MyDrive/sdar_normal_sft",     # notebook 03
        "dense500":  "/content/drive/MyDrive/sdar_superviseall",   # notebook 02
        "dense1500": "/content/drive/MyDrive/sdar_dense_matched",  # notebook 07
    })
    # source of the ORIGINAL 40-problem benchmark, so the 100 is a superset of it
    seed_bench_root: str = "/content/drive/MyDrive/sdar_normal_sft"

    model_id: str = "JetLM/SDAR-4B-Chat-b32"
    block_size: int = 32

    # LoRA fields are unused here (this notebook only LOADS adapters, never creates them),
    # but section 6 is reused verbatim from notebook 03 and its fresh_lora() helper
    # references them at definition time. Values match the training runs.
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_targets: tuple = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
    stages: Tuple[str, ...] = ("s1_t4", "s2_t2", "s3_t1")
    bench_schedules: Tuple[int, ...] = (2, 1)     # t=2 first: it is ~2x faster, fails faster
    gen_budget: int = 1024
    repeat4_max: float = 0.90

    # ---- benchmark size ----
    n_problems: int = 100
    tier_split: Tuple[int, int, int] = (50, 25, 25)   # easy, medium, hard
    dataset_id: str = "zwhe99/DeepMath-103K"
    seed: int = 0

    # ---- run control (the key to surviving Colab timeouts) ----
    only_arms: Optional[Tuple[str, ...]] = None   # e.g. ("raw",) or ("masked",) to do one at a time
    only_schedules: Optional[Tuple[int, ...]] = None

cfg = Cfg()
os.makedirs(cfg.out_root, exist_ok=True)
os.makedirs(os.path.join(cfg.out_root, "results"), exist_ok=True)
ROWS_PATH = os.path.join(cfg.out_root, "results", "rows_dense.jsonl")
BENCH_PATH = os.path.join(cfg.out_root, "bench100.json")

def ck_of(arm, stage):
    return os.path.join(cfg.drive_roots[arm], "ckpt", stage)

# ---------------------------------------------------------------------------
# Compatibility shims for section 6, which is reused BYTE-IDENTICALLY from the
# training notebooks. That cell defines training bookkeeping (save_adapters,
# adapters_exist, load_state/save_state) which this benchmark notebook never
# calls: adapters are loaded via ck_of() above, and nothing is ever saved.
# But the cell references these names at execution time, so they must exist.
# Pointing them at out_root keeps every training run's state file untouched.
# ---------------------------------------------------------------------------
STATE_PATH = os.path.join(cfg.out_root, "bench_state_unused.json")
def ck(n):  return os.path.join(cfg.out_root, "ckpt_unused", n)
def rs(n):  return os.path.join(cfg.out_root, "results", n)

ARMS = [(a, st) for a in ("dense500", "dense1500") for st in cfg.stages]
if cfg.only_arms:
    ARMS = [(a, st) for (a, st) in ARMS if a in cfg.only_arms]
SCHEDULES = list(cfg.only_schedules) if cfg.only_schedules else list(cfg.bench_schedules)

_t = {1: 1.50, 2: 0.83}   # measured min/problem from your earlier runs
_est = sum(_t[s] for s in SCHEDULES) * cfg.n_problems * len(ARMS) / 60
print(f"configs to run : {len(ARMS)}  {[a if st is None else f'{a}/{st}' for a,st in ARMS]}")
print(f"schedules      : {SCHEDULES}")
print(f"problems       : {cfg.n_problems}")
print(f"rollouts       : {len(ARMS)*len(SCHEDULES)*cfg.n_problems}")
print(f"\nestimated GPU time for THIS invocation: ~{_est:.1f} h")
print("Everything is checkpointed per rollout; re-run to resume. Use cfg.only_arms to split")
print("the work across sessions (recommended: one arm per session).")

## 4 · Preflight — verify every adapter exists before spending GPU time

A missing or misnamed checkpoint discovered 3 hours in is the expensive failure mode. This cell
checks all nine adapter directories up front and refuses to continue if any is absent.

In [ ]:
import os, json

missing, found = [], []
for arm, stage in ARMS:
    if stage is None:
        found.append("raw (base model, no adapter)"); continue
    p = ck_of(arm, stage)
    ok = os.path.isdir(p) and any(f.startswith("adapter_model") for f in os.listdir(p))
    (found if ok else missing).append(f"{arm}/{stage}  ->  {p}")

print("FOUND:")
for f in found: print("  ", f)
if missing:
    print("\nMISSING:")
    for m in missing: print("  ", m)
    raise FileNotFoundError(
        f"{len(missing)} adapter(s) not found. Fix cfg.drive_roots in section 3, or drop the "
        "arm from cfg.only_arms. Nothing has been computed yet, so this costs you nothing.")
print("\nAll adapters present. Safe to proceed.")

## 5 · Build the 100-problem benchmark (superset of the original 40)

Loads the original 40 from your masked-only run and adds 60 more from DeepMath-103K, stratified
to reach 50 easy / 25 medium / 25 hard, excluding anything already in the 40 and anything used
for training. Cached to `bench100.json` on first run and reused afterwards, so every arm scores
the identical set.

The `in_original_40` flag is preserved per problem. That lets §11 report results on the original
subset and the full 100 separately, which is the cleanest way to show the extra problems did not
shift conclusions.

In [ ]:
import json, os, random
from collections import Counter

if os.path.exists(BENCH_PATH):
    bench = json.load(open(BENCH_PATH))
    print(f"loaded cached benchmark: {len(bench)} problems")
else:
    seed_path = os.path.join(cfg.seed_bench_root, "data_partitions.json")
    orig = []
    if os.path.exists(seed_path):
        D = json.load(open(seed_path))
        orig = D.get("bench", [])
        print(f"seeded from original benchmark: {len(orig)} problems")
    else:
        print(f"WARNING: {seed_path} not found; building a fresh 100 (NOT a superset of the 40)")

    for p in orig:
        p["in_original_40"] = True
    have = {p["question"] for p in orig}
    train_q = set()
    if os.path.exists(seed_path):
        D = json.load(open(seed_path))
        for k in ("stage_a", "stage_b", "stage_c"):
            for p in D.get(k, []): train_q.add(p["question"])

    need = dict(zip(("easy", "medium", "hard"), cfg.tier_split))
    for p in orig:
        t = p.get("tier")
        if t in need: need[t] -= 1
    print("additional problems needed per tier:", need)

    if sum(max(0, v) for v in need.values()) > 0:
        from datasets import load_dataset
        raw = load_dataset(cfg.dataset_id, split="train")
        Q = "question" if "question" in raw.column_names else "problem"
        A = "final_answer" if "final_answer" in raw.column_names else "answer"
        SOLS = [c for c in ("r1_solution_1","r1_solution_2","r1_solution_3") if c in raw.column_names]
        def tier_of(d):
            d = float(d)
            return "easy" if d <= 4.0 else ("medium" if d <= 6.0 else "hard")
        order = list(range(len(raw))); random.Random(cfg.seed + 4242).shuffle(order)
        for i in order:
            if all(v <= 0 for v in need.values()): break
            ex = raw[i]; q = ex[Q]
            if q in have or q in train_q: continue
            t = tier_of(ex[cfg.__class__.__dict__.get("_diff", "difficulty")]
                        if False else ex["difficulty"])
            if need.get(t, 0) <= 0: continue
            gold = str(ex[A])
            if not gold or gold.lower() == "none": continue
            orig.append({"question": q, "gold": gold, "tier": t, "in_original_40": False})
            have.add(q); need[t] -= 1
        del raw

    bench = orig
    json.dump(bench, open(BENCH_PATH, "w"), indent=1)
    print(f"built and cached: {len(bench)} problems -> {BENCH_PATH}")

# normalize the answer key so the scorer always finds it
for p in bench:
    if "gold" not in p:
        p["gold"] = str(p.get("answer", p.get("final_answer", "")))
    p.setdefault("in_original_40", False)

print("tiers    :", dict(Counter(p["tier"] for p in bench)))
print("original :", sum(p["in_original_40"] for p in bench),
      "| new:", sum(not p["in_original_40"] for p in bench))
assert len(bench) >= cfg.n_problems, f"only {len(bench)} problems available"
bench = bench[:cfg.n_problems]

# ---- CRITICAL for the split run: both notebooks must score the IDENTICAL set ----
import hashlib
BENCH_HASH = hashlib.sha256(
    "\n".join(p["question"] for p in bench).encode("utf-8")).hexdigest()[:16]
print(f"\nBENCH HASH: {BENCH_HASH}")
print("This must MATCH in both notebook 08a and 08b. If the hashes differ, the two halves")
print("scored different problems and their results CANNOT be combined. Both notebooks read")
print(f"the same cached file at {BENCH_PATH}, so a mismatch means one of them built its own")
print("before the other had written the cache: delete the newer file and re-run that half.")
_hp = os.path.join(cfg.out_root, "bench100.hash")
if os.path.exists(_hp):
    _prev = open(_hp).read().strip()
    assert _prev == BENCH_HASH, (
        f"BENCH MISMATCH: cached hash {_prev} != this run's {BENCH_HASH}. "
        "The two notebook halves are not scoring the same problems. Stop and reconcile.")
    print("verified against bench100.hash: MATCH")
else:
    open(_hp, "w").write(BENCH_HASH)
    print("wrote bench100.hash (the other notebook will verify against this)")

## 7 · Model, LoRA, mask
`modeling_sdar.py`'s outer `forward()` has `_update_causal_mask` commented out, so the block-causal
mask is entirely our responsibility. `fuse_cross_entropy` is forced off — it returns `logits=None`
whenever `self.training` is True.

In [ ]:
import torch, contextlib, gc, math, time, transformers
from packaging import version as _v
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel

_dt_kw = "dtype" if _v.parse(transformers.__version__.split("+")[0]) >= _v.parse("4.56.0") \
         else "torch_dtype"

def build_block_causal_mask(seq_len, prompt_len, block_size, device):
    idx = torch.arange(seq_len, device=device)
    is_resp = idx >= prompt_len
    blk = torch.where(is_resp, (idx-prompt_len)//block_size, idx)
    qr,kr = is_resp.unsqueeze(1), is_resp.unsqueeze(0)
    qi,ki = idx.unsqueeze(1), idx.unsqueeze(0)
    qb,kb = blk.unsqueeze(1), blk.unsqueeze(0)
    return ((~qr)&(ki<=qi)) | (qr&(~kr)) | (qr&kr&(kb<=qb))

def load_base():
    tok = AutoTokenizer.from_pretrained(cfg.model_id, trust_remote_code=True)
    m = AutoModelForCausalLM.from_pretrained(
        cfg.model_id, trust_remote_code=True, device_map="cuda",
        attn_implementation="flash_attention_2" if USE_FLASH_ATTN else "sdpa",
        **{_dt_kw: torch.bfloat16})
    if hasattr(m.config,"fuse_cross_entropy"): m.config.fuse_cross_entropy = False
    m.gradient_checkpointing_disable()
    return m, tok

def fresh_lora(m):
    return get_peft_model(m, LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_targets), bias="none", task_type="CAUSAL_LM"))

def build_invalid(m, tok):
    vs, rv = m.config.vocab_size, len(tok)
    inv = torch.zeros(vs, dtype=torch.bool)
    if rv < vs: inv[rv:] = True
    mid = getattr(tok,"mask_token_id",None) or 151669
    inv[mid] = True
    return inv.to(m.device), mid

def student_prompt(tok, q):
    return tok.apply_chat_template(
        [{"role":"user","content": f"{q}\nPlease reason step by step, and put your final answer "
                                   f"within \\boxed{{}}."}], tokenize=False, add_generation_prompt=True)

def teacher_prompt(tok, q, sol):
    c = (f"{q}\n\nHere is a reference solution:\n{sol}\n\nAfter understanding the reference "
         f"solution, solve the problem yourself.\nPlease reason step by step, and put your final "
         f"answer within \\boxed{{}}.")
    return tok.apply_chat_template([{"role":"user","content":c}], tokenize=False,
                                   add_generation_prompt=True)

def logits_of(model, ids, prompt_len, teacher=False):
    mask = build_block_causal_mask(ids.shape[1], prompt_len, cfg.block_size, ids.device)[None,None]
    ctx = model.disable_adapter() if teacher else contextlib.nullcontext()
    with ctx:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            return model(input_ids=ids, attention_mask=mask).logits

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
    print(f"    VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

def save_adapters(model, name):
    d = ck(name); os.makedirs(d, exist_ok=True); model.save_pretrained(d)

def adapters_exist(name):
    return os.path.isfile(os.path.join(ck(name),"adapter_model.safetensors")) or \
           os.path.isfile(os.path.join(ck(name),"adapter_model.bin"))

DEFAULT_STATE = {s:{"done":False,"epoch":0,"sample":0,"opt_step":0}
                 for s in ("stage_a","stage_b","stage_c","opsd1","opsd2","bench")}
def load_state():
    if os.path.isfile(STATE_PATH):
        st = json.load(open(STATE_PATH))
        for k,v in DEFAULT_STATE.items(): st.setdefault(k, dict(v))
        return st
    return {k:dict(v) for k,v in DEFAULT_STATE.items()}
def save_state(st): json.dump(st, open(STATE_PATH,"w"), indent=1)

STATE = load_state()
print("model helpers ready.")


## 7 · Scorer and rollout — identical to the training notebooks' harness

`score_generation`, `bench_rollout` and the stop rules are copied verbatim from notebook 03's
§13 so these numbers are directly comparable to your existing n=40 results. Nothing about the
evaluation changed; only the problem count and the record-keeping did.

In [ ]:
import torch, time, json, re

def score_generation(text, gold):
    def norm_ci(s):
        if s is None: return None
        s = str(s).strip().lower().replace(" ", "")
        for a,b in (("\\left",""),("\\right",""),("\\dfrac","\\frac"),("\\tfrac","\\frac"),("$","")):
            s = s.replace(a,b)
        if s.startswith("\\text{") and s.endswith("}"): s = s[6:-1]
        return s
    def match(a,b):
        a,b = norm_ci(a), norm_ci(b)
        if a is None or b is None: return False
        if a == b: return True
        try: return abs(float(a)-float(b)) < 1e-6
        except Exception: return False
    pred = extract_boxed(text)
    strict_ok = match(pred, gold) if pred else False
    if pred is None:
        tail = text[-300:]
        for pat in (r"(?:the answer is|answer:)\s*\$?([^\.\n$]{1,40})",
                    r"(?:the limit is|equals?)\s*\$?([^\.\n$]{1,40})",
                    r"=\s*\$?([^\.\n$=]{1,25})\s*\$?\.?\s*$"):
            m = re.findall(pat, tail, re.IGNORECASE)
            if m: pred = m[-1].strip(); break
    return {"pred": pred, "correct": match(pred, gold) if pred else False,
            "correct_strict": strict_ok}

@torch.no_grad()
def bench_rollout(model, tok, invalid, mask_id, question, tps):
    dev = model.device
    ids = tok(student_prompt(tok, question), return_tensors="pt").input_ids.to(dev)
    plen = ids.shape[1]; gen, txt, why = 0, "", "budget"
    while gen < cfg.gen_budget:
        cur = ids.shape[1]
        work = torch.cat([ids, torch.full((1,cfg.block_size), mask_id, dtype=ids.dtype, device=dev)], 1)
        pos = torch.arange(cur, cur+cfg.block_size, device=dev)
        still = torch.ones(cfg.block_size, dtype=torch.bool, device=dev)
        while bool(still.any()):
            lg = logits_of(model, work, plen)[0,pos,:].float().masked_fill(invalid, float("-inf"))
            conf = torch.log_softmax(lg,-1).max(-1).values.masked_fill(~still, float("-inf"))
            k = min(tps, int(still.sum().item()))
            sel = conf.topk(k).indices
            work[0,pos[sel]] = lg[sel].argmax(-1); still[sel] = False
        new = work[:,cur:cur+cfg.block_size]
        ids = torch.cat([ids,new],1); gen += cfg.block_size
        g = [t for t in ids[0,plen:].tolist() if t < len(tok)]
        txt = tok.decode(g, skip_special_tokens=True)
        if has_complete_box(txt): why="box"; break
        if tok.eos_token_id is not None and bool((new==tok.eos_token_id).any()): why="eos"; break
        if gen >= 128 and repeat4(txt) > cfg.repeat4_max: why="collapse"; break
    return txt, gen, why

print("scorer + rollout ready (verbatim from notebook 03 section 13)")

## 8 · Main loop — resumable at **per-rollout** granularity

Every completed rollout appends one line to `per_problem_rows.jsonl` immediately. On restart the
notebook reads that file and skips every `(arm, stage, schedule, problem)` already present, so a
disconnect costs at most the single rollout in flight.

Each model is loaded once, scored on all its schedules and problems, then freed before the next
model loads. That keeps peak VRAM at one model regardless of how many arms are in the run.

**To split across sessions**, set `cfg.only_arms` in §3 and re-run the notebook. For example
`("raw",)`, then `("masked",)`, then `("dense500",)`, then `("dense1500",)`.

In [ ]:
import os, json, gc, time, torch
from peft import PeftModel

done = set()
if os.path.exists(ROWS_PATH):
    for line in open(ROWS_PATH):
        try:
            r = json.loads(line)
            done.add((r["arm"], r["stage"], r["tps"], r["pidx"]))
        except Exception:
            pass
    print(f"resuming: {len(done)} rollouts already recorded")

todo = [(a, st, t, i) for (a, st) in ARMS for t in SCHEDULES
        for i in range(len(bench))
        if (a, st or "-", t, i) not in done]
print(f"remaining this invocation: {len(todo)} rollouts "
      f"(~{sum(1.5 if t==1 else 0.83 for _,_,t,_ in todo)/60:.1f} h)\n")

fh = open(ROWS_PATH, "a")
t_start = time.time()
for arm, stage in ARMS:
    pend = [(t, i) for t in SCHEDULES for i in range(len(bench))
            if (arm, stage or "-", t, i) not in done]
    if not pend:
        print(f"=== {arm}/{stage or 'base'}: already complete, skipping ===")
        continue

    print(f"\n=== loading {arm}/{stage or 'base'} ===")
    base, tok = load_base()
    if stage is None:
        model = base
    else:
        peft = PeftModel.from_pretrained(base, ck_of(arm, stage), is_trainable=False)
        model = peft.merge_and_unload()
    model.gradient_checkpointing_disable()
    if hasattr(model.config, "fuse_cross_entropy"): model.config.fuse_cross_entropy = False
    invalid, mask_id = build_invalid(model, tok); model.eval()

    for tps in SCHEDULES:
        idxs = [i for (t, i) in pend if t == tps]
        if not idxs: continue
        t0, nc = time.time(), 0
        for c, i in enumerate(idxs, 1):
            p = bench[i]
            txt, n, why = bench_rollout(model, tok, invalid, mask_id, p["question"], tps)
            sc = score_generation(txt, p["gold"])
            row = {"arm": arm, "stage": stage or "-", "tps": tps, "pidx": i,
                   "tier": p["tier"], "in_original_40": p["in_original_40"],
                   "correct": bool(sc["correct"]), "correct_strict": bool(sc["correct_strict"]),
                   "pred": sc["pred"], "gold": p["gold"],
                   "repeat4": float(repeat4(txt)), "stop": why, "gen_tokens": n}
            fh.write(json.dumps(row) + "\n"); fh.flush()
            done.add((arm, stage or "-", tps, i))
            nc += int(sc["correct"])
            if c % 10 == 0 or c == len(idxs):
                el = (time.time() - t0) / 60
                print(f"    {arm}/{stage or 'base'} t{tps}: {c}/{len(idxs)} | "
                      f"correct {nc}/{c} | {el:.1f} min | eta {el/c*(len(idxs)-c):.1f} min")
        print(f"  DONE {arm}/{stage or 'base'} t{tps}: {nc}/{len(idxs)} = {nc/max(len(idxs),1):.1%}")

    del model, base, tok, invalid
    if stage is not None:
        try: del peft
        except Exception: pass
    gc.collect(); torch.cuda.empty_cache()
    print(f"  freed | total elapsed {(time.time()-t_start)/60:.1f} min")

fh.close()
print(f"\nALL DONE for this invocation -> {ROWS_PATH}")

## 9 · Results table with bootstrap confidence intervals

Reads the JSONL and aggregates. Because per-problem records now exist, every number gets a real
95% bootstrap interval rather than a reconstructed one. The table also reports the original-40
subset alongside the full 100 so you can show directly that the larger benchmark did not move
the conclusions.

In [ ]:
import json, random, statistics
from collections import defaultdict

import glob
rows = []
_files = sorted(glob.glob(os.path.join(cfg.out_root, "results", "rows_*.jsonl")))
for _f in _files:
    n0 = len(rows)
    for l in open(_f):
        try: rows.append(json.loads(l))
        except Exception: pass
    print(f"  {os.path.basename(_f):<28} {len(rows)-n0:>5} rollouts")
print(f"\n{len(rows)} rollouts total across {len(_files)} file(s)")
_have = {(r["arm"], r["stage"]) for r in rows}
_want = {("raw","-")} | {(a,s) for a in ("masked","dense500","dense1500") for s in cfg.stages}
_miss = sorted(_want - _have)
if _miss:
    print(f"NOTE: {len(_miss)} config(s) not yet present (run the other notebook): {_miss}")
print()

def boot_ci(vals, n=10000, seed=0):
    if not vals: return (float("nan"), float("nan"))
    random.seed(seed); m = len(vals)
    b = sorted(sum(random.choice(vals) for _ in range(m))/m for _ in range(n))
    return b[int(.025*n)], b[int(.975*n)]

def agg(sel):
    c = [float(r["correct"]) for r in sel]
    if not c: return None
    lo, hi = boot_ci(c)
    d = {"n": len(c), "acc": sum(c)/len(c), "ci": (lo, hi),
         "strict": sum(float(r["correct_strict"]) for r in sel)/len(sel),
         "rep4": sum(r["repeat4"] for r in sel)/len(sel)}
    for tier in ("easy", "medium", "hard"):
        tr = [float(r["correct"]) for r in sel if r["tier"] == tier]
        d[tier] = sum(tr)/len(tr) if tr else float("nan")
    for st in ("box", "eos", "collapse", "budget"):
        d["stop_"+st] = sum(1 for r in sel if r["stop"] == st)/len(sel)
    return d

LABEL = {("raw","-"):"raw", ("masked","s1_t4"):"masked s1", ("masked","s2_t2"):"masked s2",
         ("masked","s3_t1"):"masked s3", ("dense500","s1_t4"):"dense500 s1",
         ("dense500","s2_t2"):"dense500 s2", ("dense500","s3_t1"):"dense500 s3",
         ("dense1500","s1_t4"):"dense1500 s1", ("dense1500","s2_t2"):"dense1500 s2",
         ("dense1500","s3_t1"):"dense1500 s3"}
ORDER = list(LABEL.keys())

summary = {}
for tps in sorted({r["tps"] for r in rows}):
    print(f"{'='*96}\ntokens_per_step = {tps}   (n = {cfg.n_problems})")
    print(f"{'model':<15}{'n':>4}{'acc':>7}{'95% CI':>16}{'strict':>8}{'easy':>7}{'med':>7}{'hard':>7}{'rep4':>7}{'collapse':>9}")
    for key in ORDER:
        sel = [r for r in rows if (r["arm"], r["stage"]) == key and r["tps"] == tps]
        d = agg(sel)
        if not d: continue
        summary[f"{key[0]}|{key[1]}|t{tps}"] = d
        print(f"{LABEL[key]:<15}{d['n']:>4}{d['acc']:>7.1%}"
              f"   [{d['ci'][0]:.1%},{d['ci'][1]:.1%}]".rjust(16)
              + f"{d['strict']:>8.1%}{d['easy']:>7.1%}{d['medium']:>7.1%}{d['hard']:>7.1%}"
                f"{d['rep4']:>7.3f}{d['stop_collapse']:>9.1%}")
    print()

# ---- original-40 subset vs the new 60, as a stability check ----
print(f"{'='*96}\nSTABILITY: original 40 vs full 100 (t=1 accuracy)")
print(f"{'model':<15}{'orig40':>9}{'full100':>10}{'shift':>8}")
for key in ORDER:
    a = [float(r["correct"]) for r in rows
         if (r["arm"], r["stage"]) == key and r["tps"] == 1 and r["in_original_40"]]
    b = [float(r["correct"]) for r in rows
         if (r["arm"], r["stage"]) == key and r["tps"] == 1]
    if not a or not b: continue
    print(f"{LABEL[key]:<15}{sum(a)/len(a):>9.1%}{sum(b)/len(b):>10.1%}{sum(b)/len(b)-sum(a)/len(a):>+8.1%}")

json.dump({k: {kk: (list(vv) if isinstance(vv, tuple) else vv) for kk, vv in v.items()}
           for k, v in summary.items()},
          open(os.path.join(cfg.out_root, "results", "summary_100.json"), "w"), indent=1)
print("\nwrote summary_100.json")

## 10 · Significance tests — do the comparisons resolve at n=100?

The comparisons that mattered and failed at n=40:

* `masked` vs `dense1500` at each stage: **the supervision-density effect on matched data.**
* `dense1500` vs `dense500` at each stage: **the data-cleanliness effect.**

At n=40 only s2 cleared significance and the rest needed pooling. This cell reports each
comparison individually at n=100 plus the pooled version, so you can lead with per-stage results
if they now resolve.

In [ ]:
import math, json

def two_prop(k1, n1, k2, n2):
    if n1 == 0 or n2 == 0: return float("nan"), float("nan")
    p1, p2 = k1/n1, k2/n2
    pp = (k1+k2)/(n1+n2)
    se = math.sqrt(pp*(1-pp)*(1/n1 + 1/n2))
    if se == 0: return 0.0, 1.0
    z = (p1-p2)/se
    return z, 2*(1 - 0.5*(1 + math.erf(abs(z)/math.sqrt(2))))

def counts(arm, stage, tps):
    sel = [r for r in rows if r["arm"] == arm and r["stage"] == stage and r["tps"] == tps]
    return sum(r["correct"] for r in sel), len(sel)

for tps in sorted({r["tps"] for r in rows}):
    print(f"{'='*88}\ntokens_per_step = {tps}")
    for label, A, B in [("supervision density (masked vs dense1500)", "masked", "dense1500"),
                        ("data cleanliness (dense1500 vs dense500)", "dense1500", "dense500")]:
        print(f"\n  {label}")
        tot = [0, 0, 0, 0]
        for st in cfg.stages:
            k1, n1 = counts(A, st, tps); k2, n2 = counts(B, st, tps)
            if n1 == 0 or n2 == 0: continue
            z, p = two_prop(k1, n1, k2, n2)
            tot[0] += k1; tot[1] += n1; tot[2] += k2; tot[3] += n2
            flag = "SIG" if p < 0.05 else "ns "
            print(f"    {st}: {A} {k1}/{n1}={k1/n1:.1%}  vs  {B} {k2}/{n2}={k2/n2:.1%}   "
                  f"p={p:.4f}  {flag}")
        if tot[1] and tot[3]:
            z, p = two_prop(tot[0], tot[1], tot[2], tot[3])
            flag = "SIG" if p < 0.05 else "ns "
            print(f"    POOLED: {tot[0]}/{tot[1]}={tot[0]/tot[1]:.1%}  vs  "
                  f"{tot[2]}/{tot[3]}={tot[2]/tot[3]:.1%}   p={p:.4f}  {flag}")
    print()
print("Note: pooling assumes the effect is stage-invariant. If per-stage tests now resolve on")
print("their own, lead with those and use the pooled number only as a supporting check.")

## 11 · Figures

Four panels, all written to `results/` at 200 dpi and ready to drop into the paper:

1. **Accuracy by arm and stage, with 95% CIs** — the headline comparison.
2. **Per-tier breakdown** — where the damage lands (medium tier is the sensitive one).
3. **Repeat-4** — the degeneration signature.
4. **Stop reasons** — how generations end (box / eos / collapse / budget).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, os, json

FIGDIR = os.path.join(cfg.out_root, "results")
COL = {"raw": "0.35", "masked": "#4477aa", "dense500": "#ee7733", "dense1500": "#cc3311"}

def cells(tps):
    out = []
    for key in ORDER:
        k = f"{key[0]}|{key[1]}|t{tps}"
        if k in summary: out.append((key, summary[k]))
    return out

# ---- 1. accuracy with CIs ----
fig, axes = plt.subplots(1, len(SCHEDULES), figsize=(6.2*len(SCHEDULES), 3.6), squeeze=False)
for ax, tps in zip(axes[0], sorted({r["tps"] for r in rows})):
    cs = cells(tps)
    x = np.arange(len(cs))
    acc = [d["acc"] for _, d in cs]
    lo = [d["acc"] - d["ci"][0] for _, d in cs]
    hi = [d["ci"][1] - d["acc"] for _, d in cs]
    ax.bar(x, acc, yerr=[lo, hi], capsize=3,
           color=[COL[k[0]] for k, _ in cs], edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels([LABEL[k] for k, _ in cs], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("accuracy"); ax.set_title(f"tokens/step = {tps}  (n={cfg.n_problems}, 95% CI)")
    ax.axhline(summary.get(f"raw|-|t{tps}", {}).get("acc", 0), ls=":", c="0.4", lw=1)
    ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig_accuracy_100.png", dpi=200, bbox_inches="tight"); plt.show()

# ---- 2. per-tier ----
fig, axes = plt.subplots(1, len(SCHEDULES), figsize=(6.2*len(SCHEDULES), 3.6), squeeze=False)
for ax, tps in zip(axes[0], sorted({r["tps"] for r in rows})):
    cs = cells(tps); x = np.arange(len(cs)); w = 0.26
    for off, tier, c in [(-w, "easy", "#4477aa"), (0, "medium", "#ee7733"), (w, "hard", "#cc3311")]:
        ax.bar(x+off, [d[tier] for _, d in cs], w, label=tier, color=c, edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels([LABEL[k] for k, _ in cs], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("accuracy"); ax.set_title(f"by difficulty tier, t={tps}")
    ax.legend(frameon=False, fontsize=8); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig_tiers_100.png", dpi=200, bbox_inches="tight"); plt.show()

# ---- 3. repeat-4 ----
fig, ax = plt.subplots(figsize=(7.2, 3.4))
for tps, mk in zip(sorted({r["tps"] for r in rows}), ("o-", "s--")):
    cs = cells(tps)
    ax.plot(range(len(cs)), [d["rep4"] for _, d in cs], mk, label=f"t={tps}", markersize=5)
ax.set_xticks(range(len(cells(1) or cells(2))))
ax.set_xticklabels([LABEL[k] for k, _ in (cells(1) or cells(2))], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("repeat-4 (lower better)"); ax.set_title("degeneration signature")
ax.legend(frameon=False); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig_repeat4_100.png", dpi=200, bbox_inches="tight"); plt.show()

# ---- 4. stop reasons ----
fig, axes = plt.subplots(1, len(SCHEDULES), figsize=(6.2*len(SCHEDULES), 3.6), squeeze=False)
for ax, tps in zip(axes[0], sorted({r["tps"] for r in rows})):
    cs = cells(tps); x = np.arange(len(cs)); bottom = np.zeros(len(cs))
    for st, c in [("box","#4477aa"),("eos","#88ccee"),("collapse","#cc3311"),("budget","0.7")]:
        v = np.array([d["stop_"+st] for _, d in cs])
        ax.bar(x, v, bottom=bottom, label=st, color=c, edgecolor="white"); bottom += v
    ax.set_xticks(x); ax.set_xticklabels([LABEL[k] for k, _ in cs], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("fraction"); ax.set_title(f"stop reason, t={tps}")
    ax.legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig(f"{FIGDIR}/fig_stops_100.png", dpi=200, bbox_inches="tight"); plt.show()

print("wrote 4 figures to", FIGDIR)

## 12 · LaTeX table rows

Prints paper-ready rows with CIs so nothing has to be transcribed by hand. Transcription is
where table errors come from.

In [ ]:
for tps in sorted({r["tps"] for r in rows}):
    print(f"% ---- tokens/step = {tps}, n = {cfg.n_problems} ----")
    for key in ORDER:
        k = f"{key[0]}|{key[1]}|t{tps}"
        if k not in summary: continue
        d = summary[k]
        print(f"    {LABEL[key]:<15} & {d['acc']*100:>4.1f} "
              f"\\tiny[{d['ci'][0]*100:.0f},{d['ci'][1]*100:.0f}] "
              f"& {d['easy']*100:>4.1f} & {d['medium']*100:>4.1f} & {d['hard']*100:>4.1f} "
              f"& {d['rep4']:.3f} \\\\")
    print()

## 13 · Notes

**Running this across sessions.** Set `cfg.only_arms` in §3 to one arm and run the whole
notebook; repeat per arm. The JSONL accumulates across sessions and §9–§12 always analyse
everything recorded so far, so partial results are still usable and the figures still render.

**Reporting.** Lead with the n=100 numbers. Keep the n=40 results in the paper only if you need
to show consistency; §9's stability panel gives you the shift per arm, and a small shift is
itself worth one sentence, since it shows the original conclusions were not an artifact of a
lucky 40-problem draw.

**If a comparison still fails to resolve at n=100**, say so plainly rather than pooling to
manufacture significance. A clearly-reported null at n=100 is a stronger contribution than a
borderline positive at n=40, and reviewers respond much better to it.

**Do not re-run arms that finished.** The resume logic protects you automatically, but deleting
`per_problem_rows.jsonl` throws away all 39 hours. Back it up.